![image_1781182316433.png](./image_1781182316433.png "image_1781182316433.png")

![image_1781182355104.png](./image_1781182355104.png "image_1781182355104.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import Window
# Initialize Spark session
spark = SparkSession.builder.appName("SalaryHistoryDataFrame").getOrCreate()

# Define the dataset
data = [
    (1, 1001, 65000, "2022-01-01"),
    (2, 1001, 72000, "2023-01-01"),
    (3, 1001, 80000, "2024-01-01"),
    (4, 1002, 90000, "2022-06-01"),
    (5, 1002, 88000, "2023-06-01"),
    (6, 1003, 55000, "2023-03-01"),
    (7, 1003, 62000, "2024-03-01"),
    (8, 1004, 110000, "2022-09-01"),
    (9, 1004, 115000, "2023-09-01"),
    (10, 1004, 125000, "2024-09-01"),
    (11, 1005, 78000, "2023-11-01")
]

# Define schema
columns = ["id", "employee_id", "salary", "effective_date"]

# Create DataFrame
salary_df = spark.createDataFrame(data, columns)

# Show DataFrame
salary_df.show()


In [0]:
salary_df = salary_df.withColumn(
    "rank",
    f.dense_rank().over(
        Window.partitionBy("employee_id").orderBy(f.col("effective_date").desc())
    ),
)

s1_df = salary_df.filter(f.col("rank") == 1)
s2_df = salary_df.filter(f.col("rank") == 2)

result_df = (
    s1_df.join(s2_df, s1_df.employee_id == s2_df.employee_id)
    .filter(s1_df.salary > s2_df.salary)
    .select(
        s1_df.employee_id.alias("employee_id"),
        s2_df.salary.alias("previous_salary"),
        s1_df.salary.alias("current_salary"),
        (s1_df.salary - s2_df.salary).alias("increase_amount"),
    )
    .orderBy(f.col("increase_amount").desc())
)
display(result_df)

In [0]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window

# Add rank column
salary_df = salary_df.withColumn(
    "rank",
    f.dense_rank().over(
        Window.partitionBy("employee_id").orderBy(f.col("effective_date").desc())
    ),
)

# Self-join directly on rank values
result_df = (
    salary_df.alias("curr")
    .join(
        salary_df.alias("prev"),
        (f.col("curr.employee_id") == f.col("prev.employee_id"))
        & (f.col("curr.rank") == 1)
        & (f.col("prev.rank") == 2),
    )
    .filter(f.col("curr.salary") > f.col("prev.salary"))
    .select(
        f.col("curr.employee_id").alias("employee_id"),
        f.col("prev.salary").alias("previous_salary"),
        f.col("curr.salary").alias("current_salary"),
        (f.col("curr.salary") - f.col("prev.salary")).alias("increase_amount"),
    )
    .orderBy(f.col("increase_amount").desc())
)

result_df.show()
